# Neeko TTS Demo — Chatterbox Multilingual (Türkçe)

**Model:** Resemble AI Chatterbox Multilingual (MIT lisans, 23 dil resmi Türkçe destek)

**Donanım:** Kaggle T4 x2 GPU (Settings → Accelerator → **GPU T4 x2** seçili olmalı)

**Çıktı:** 10 Türkçe çocuk cümlesinin .wav dosyaları + tek bir `neeko-demo-v0.1.zip` paketi (sağ taraftaki **Output** sekmesinden indirilir).

**Tahmini süre:** ~8-12 dakika (model indirme + 10 cümle sentezleme).

**Repo:** [`neeko-voice/notebooks/00-chatterbox-tr-demo.ipynb`](https://github.com/) — versiyon kontrolünde.

## 1. Kurulum
Chatterbox-TTS paketini ve gerekli ses kütüphanelerini yükle. İlk koşuda ~30 saniye sürer.

In [ ]:
!pip install -q chatterbox-tts

## 2. Donanım doğrulama
GPU gerçekten aktif mi? Output `cuda` görmeli.

In [ ]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM (toplam): {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("UYARI: GPU yok. Settings -> Accelerator -> GPU T4 x2 seç ve session'ı restart et.")

## 3. Model yükle
Chatterbox Multilingual modelini Hugging Face'ten indirip GPU'ya yükle. İlk koşuda ~3-6 dakika sürer (model dosyaları yaklaşık 1-2 GB).

In [ ]:
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
import time

t0 = time.time()
model = ChatterboxMultilingualTTS.from_pretrained(device=device)
print(f"Model yüklendi: {time.time() - t0:.1f} saniye")
print(f"Sample rate: {model.sr} Hz")

## 4. Test cümleleri
Neeko mini eval seti v0.1 — `data/test-sets/v0.1-mini.md` repo'daki sürümle aynı. 10 cümle, her biri farklı child-directed kategori (oyun, uyku, ders, şefkat, soru, heyecan, sayı, kısaltma, kod-karışımı, hikaye).

In [ ]:
test_sentences = [
    ("01", "oyun",        "Hadi birlikte sıradaki hayvanı bulalım!"),
    ("02", "uyku",        "Şimdi gözlerini kapatıp derin bir nefes alalım."),
    ("03", "ders",        "Yedi artı üç kaç eder, biliyor musun?"),
    ("04", "sefkat",      "Kızgın olman çok normal, önce sakinleşelim."),
    ("05", "soru",        "Sence bugün ne renk bir gökyüzü gördük?"),
    ("06", "heyecan",     "Vay canına, çok güzel bir resim çizmişsin!"),
    ("07", "sayi_tarih",  "Yirmi üç Nisan'da ne kutlarız?"),
    ("08", "kisaltma",    "Doktor Ayşe, bunu bir büyüğünle birlikte yapalım dedi."),
    ("09", "kod_karisim", "Annenin iPhone'unu yerine bırakır mısın?"),
    ("10", "hikaye",      "Bir varmış, bir yokmuş, çok uzak bir ülkede küçük bir tavşan yaşarmış."),
]

for idx, cat, text in test_sentences:
    print(f"{idx} [{cat:12s}] {text}")

## 5. Sentezle ve kaydet
Her cümleyi sırayla sentezle, `/kaggle/working/output/` altına `.wav` olarak kaydet. Her cümle ~3-8 saniye sürer.

In [ ]:
import os
import torchaudio as ta

out_dir = "/kaggle/working/output"
os.makedirs(out_dir, exist_ok=True)

results = []
for idx, cat, text in test_sentences:
    t0 = time.time()
    wav = model.generate(text, language_id="tr")
    elapsed = time.time() - t0
    
    out_path = f"{out_dir}/{idx}_{cat}.wav"
    ta.save(out_path, wav, model.sr)
    
    duration = wav.shape[-1] / model.sr
    rtf = elapsed / duration if duration > 0 else float('inf')
    results.append({"id": idx, "cat": cat, "text": text, "duration_s": duration, "elapsed_s": elapsed, "rtf": rtf})
    print(f"{idx} [{cat:12s}] {duration:.2f}s ses, {elapsed:.2f}s işlem, RTF={rtf:.2f}")

print(f"\nToplam: {len(results)} dosya kaydedildi -> {out_dir}")

## 6. Metadata yaz
Çıktı paketinde hangi cümle hangi dosya, RTF kaç — bunu `metadata.json` olarak ekle ki ileride karşılaştırma kolay olsun.

In [ ]:
import json
from datetime import datetime

metadata = {
    "model": "ResembleAI/chatterbox-multilingual",
    "language": "tr",
    "sample_rate_hz": int(model.sr),
    "device": device,
    "date_utc": datetime.utcnow().isoformat(),
    "test_set_version": "v0.1-mini",
    "sentences": results,
    "mean_rtf": sum(r["rtf"] for r in results) / len(results),
}

with open(f"{out_dir}/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(json.dumps(metadata, ensure_ascii=False, indent=2))

## 7. Paketle
Tüm çıktıları tek bir `neeko-demo-v0.1.zip` olarak paketle. Bu dosya sağ panelde **Output** sekmesinde görünür ve oradan indirilir.

In [ ]:
import shutil
zip_path = "/kaggle/working/neeko-demo-v0.1"
shutil.make_archive(zip_path, "zip", out_dir)
print(f"Paket hazır: {zip_path}.zip")
print("\nİndirme: Sağ panel -> Output -> neeko-demo-v0.1.zip")

## 8. Hızlı dinleme (opsiyonel)
Notebook içinden direkt dinlemek istersen bu hücreyi çalıştır. Aşağı kaydırıp her ses dosyasını oynat.

In [ ]:
from IPython.display import Audio, display, Markdown

for idx, cat, text in test_sentences:
    display(Markdown(f"**{idx} [{cat}]** — {text}"))
    display(Audio(f"{out_dir}/{idx}_{cat}.wav"))

---

## Sonraki adımlar

1. Ses dosyalarını indir, Atlas ile beraber dinle
2. Her cümle için 1-5 puan ver: anlaşılırlık, doğallık, çocuğa uygunluk, prozodi, karakter potansiyeli
3. Karşılaştırma için aynı setle **VoxCPM2** demo (`01-voxcpm2-tr-demo.ipynb`) ve **ElevenLabs** referans çıktıları üret
4. İlk değerlendirmeyi `experiments/2026-05-19-chatterbox-baseline/log.md` olarak repo'ya yaz